# 09 · Variantes
*rsLoRA, LoRA+ y DoRA: la escala, los learning rates y la forma del cambio*

Notebook del episodio 9 de **LoRA** ([ver la lección](https://d32vh5r3pvn9et.cloudfront.net/cursos/lora/variantes)). Es el mismo guion del video, en texto, con cada bloque de código para correr y su salida real.

**Mecanismo:** peft scaling: α/r vs α/√r (use_rslora) → intuición ‖BAx‖ ~ √r → r = 64 con α/r aprende casi como r = 16; con α/√r baja a 0,013 → LoRA+: dos param groups, λ = 16 llega en 50 pasos adonde LoRA en 150 → con Adam, escala ×c ≡ lr de B ×c (rsLoRA r=16 = LoRA+ λ=4, mismos números) → DoRA: m = ‖filas de W₀‖, W' = m·V/‖V‖, verificado contra use_dora=True (1,8e-7) → misma tarea, 600 pasos: al azar empatan; estirar filas (rango completo): DoRA 10× menos error

**Cómo correrlo:** con el entorno de `~/Movies/serie-lora/lab` (`cd lab && uv run --project ~/Movies/serie-alineacion/lab --group notebooks --with jupyterlab jupyter lab ../notebooks`, o en VS Code eligiendo el kernel `lab/.venv`). Todo corre offline: Gemma 2 2B sale de la caché de Hugging Face, y los entrenamientos son de juguete, con datos sintéticos.

Cada bloque de código arranca de cero (`%reset -f`): la celda de **preparación** es lo que el video no muestra, y la siguiente es exactamente el código que aparece en pantalla.

## ▶ Antes de empezar

Corré la celda de abajo una vez: baja el código de la serie y sus dependencias. Funciona en Google Colab (gratis) y en cualquier Jupyter. Para guardar tus cambios en Colab: **Archivo → Guardar una copia en Drive**.

In [ ]:
import os, subprocess, sys
SERIE = "lora"
REPO = "https://github.com/Faridmurzone/paperclase-notebooks"

def _raiz():  # carpeta del repo si ya estamos adentro
    d = os.path.abspath(os.getcwd())
    while True:
        if os.path.isfile(os.path.join(d, ".paperclase-notebooks")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

RAIZ = _raiz()
if RAIZ is None:
    RAIZ = os.path.abspath("paperclase-notebooks")
    if not os.path.isdir(RAIZ):
        subprocess.run(["git", "clone", "--depth", "1", REPO, RAIZ], check=True)
os.chdir(os.path.join(RAIZ, SERIE, "lab"))  # los bloques de código importan desde snippets/ y leen datos/
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "../requirements.txt"], check=True)
os.environ["HF_HUB_OFFLINE"] = "0"  # el notebook original corría sin red; acá baja los modelos de Hugging Face
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("listo:", os.getcwd())

In [1]:
import os
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import warnings; warnings.filterwarnings("ignore")

*LoRA. Episodio nueve: variantes. LoRA es low-rank adaptation, adaptación de rango bajo. Desde dos mil veintiuno salieron decenas de variantes. Vamos a ver tres que peft trae listas, las tres de una línea de configuración, y las vamos a implementar y comparar en la misma tarea de juguete.*

### Tres perillas de la misma fórmula

- rsLoRA: la escala, $\alpha/\sqrt{r}$ en vez de $\alpha/r$
- LoRA+: B aprende más rápido que A
- DoRA: separa magnitud y dirección
- Misma tarea sintética para las tres

Las tres tocan una pieza distinta de la misma fórmula. rsLoRA, rank-stabilized LoRA, cambia la escala: alfa sobre raíz de r, en vez de alfa sobre r. LoRA+ le da a B un learning rate más grande que a A. DoRA, weight-decomposed LoRA, separa cada fila de la matriz en un largo y una dirección, y entrena el largo aparte. Para compararlas: una capa de sesenta y cuatro por sesenta y cuatro y una tarea sintética, con Adam, adaptive moment estimation.

## Parte 1 · rsLoRA: la escala

Con alfa fijo, subir el rango achica la escala alfa sobre r. ¿Eso frena el aprendizaje?

### La escala que pone peft

Alfa fijo en ocho. Tres rangos: cuatro, dieciséis y sesenta y cuatro. use rslora es el único cambio. Y leemos la escala que peft guardó en la capa. Con LoRA común, alfa sobre r: dos, cero coma cinco, cero coma ciento veinticinco. Con rsLoRA, alfa sobre raíz de r: cuatro, dos y uno. Con rango sesenta y cuatro, la escala común es ocho veces más chica.

In [2]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep09_escalas.py
__file__ = 'snippets/ep09_escalas.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import math
import torch.nn as nn
from peft import LoraConfig, get_peft_model


class Capa(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Linear(64, 64, bias=False)

In [3]:
alpha = 8
print("   r   LoRA α/r   rsLoRA α/√r")
for r in [4, 16, 64]:
    esc = []
    for rs in [False, True]:
        cfg = LoraConfig(r=r, lora_alpha=alpha, target_modules=["proj"], use_rslora=rs)
        capa = get_peft_model(Capa(), cfg).base_model.model.proj
        esc.append(capa.scaling["default"])
    print(f"{r:4} {esc[0]:9.3f} {esc[1]:11.3f}")

   r   LoRA α/r   rsLoRA α/√r
   4     2.000       4.000
  16     0.500       2.000
  64     0.125       1.000


### Por qué la raíz de r

$$
\gamma_r = \frac{\alpha}{r}
$$
<small>$\gamma_r$: la escala · LoRA común</small>

$$
\lVert B A x \rVert \;\sim\; \sqrt{r}
$$
<small>r aportes que no se alinean: crecen como √r</small>

$$
\gamma_r \lVert BAx \rVert \sim \frac{\alpha}{r}\sqrt{r} = \frac{\alpha}{\sqrt r}\;\to\; \gamma_r = \frac{\alpha}{\sqrt r}
$$
<small>rsLoRA: que el desvío no se achique con r</small>

Gamma sub r es la escala que multiplica a B por A. Con LoRA común, alfa sobre r: en el dibujo, se achica rápido. Una intuición, no la prueba del paper. B por A por x suma r aportes, uno por columna de B. Al empezar, apuntan para cualquier lado y se cancelan en parte: la suma crece como raíz de r, no como r. En el dibujo, columnas con signos mezclados. Entonces el desvío entero va como alfa sobre r por raíz de r: alfa sobre raíz de r. Se achica al subir el rango. El paper de rsLoRA, de Kalajdzievski, de dos mil veintitrés, prueba que los adapters hay que dividirlos por la raíz del rango para que el aprendizaje no se frene.

### Episodio tres: cuánto mueve un paso, según r

Esto ya lo medimos en el episodio tres: una capa con LoRA, un solo paso de Adam, y cuánto cambia la salida, para r de uno a sesenta y cuatro. Con alfa sobre r, el cambio cae de cero coma once a cero coma cero dieciocho: seis veces menos. Subir el rango frena el primer paso. Con alfa sobre raíz de r, queda casi plano: de cero coma once a cero coma ciento cuarenta y seis. Esa es la promesa de rsLoRA: elegís alfa una vez, y cambiar r no cambia cuánto aprende cada paso.

### Tres rangos, dos escalas

La tarea: la base es una capa al azar, y el objetivo pide un cambio al azar de rango completo, así que un rango mayor debería ayudar. Tres rangos. Las dos escalas. Ciento cincuenta pasos de Adam, learning rate uno por diez a la menos tres. Miramos el paso cincuenta y el final. Con rango cuatro, casi lo mismo: cero coma cinco dos nueve contra cero coma cinco dos. Con rango sesenta y cuatro, la diferencia es enorme: con alfa sobre r termina en cero coma veinticinco; con alfa sobre raíz de r, en cero coma cero uno tres. Con la escala común, pasar de dieciséis a sesenta y cuatro casi no ayudó.

In [4]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep09_rango.py
__file__ = 'snippets/ep09_rango.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import json
import os
import torch
import torch.nn as nn
from peft import LoraConfig, get_peft_model
d = 64
DATOS = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "datos")


class Capa(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Linear(d, d, bias=False)

    def forward(self, x):
        return self.proj(x)


def tarea(tipo="azar"):
    """Base al azar y una tarea sintética: un cambio al azar de rango completo, o reescalar las filas de W."""
    torch.manual_seed(0)
    base = Capa()
    W0 = base.proj.weight.detach()
    g = torch.Generator().manual_seed(123)
    cambio = torch.randn(d, d, generator=g) * 0.1
    X = torch.randn(512, d, generator=g)
    if tipo == "azar":
        Y = X @ (W0 + cambio).T
    else:                                       # cada fila de W, estirada o encogida
        escala = torch.exp(0.3 * torch.randn(d, 1, generator=g))   # siempre > 0
        Y = X @ (escala * W0).T
    return base, X, Y


def entrenar(r=16, rs=False, dora=False, lam=1.0, lr=1e-3, pasos=150, tipo="azar"):
    base, X, Y = tarea(tipo)
    cfg = LoraConfig(r=r, lora_alpha=8, target_modules=["proj"], use_rslora=rs, use_dora=dora)
    m = get_peft_model(base, cfg)
    A = [p for n, p in m.named_parameters() if p.requires_grad and "lora_B" not in n]
    B = [p for n, p in m.named_parameters() if "lora_B" in n]
    opt = torch.optim.Adam([{"params": A, "lr": lr}, {"params": B, "lr": lam * lr}])
    hist = []
    for _ in range(pasos):
        perdida = ((m(X) - Y) ** 2).mean()
        opt.zero_grad()
        perdida.backward()
        opt.step()
        hist.append(perdida.item())
    return hist

In [5]:
finales = {}
for r in [4, 16, 64]:
    for rs in [False, True]:
        hist = entrenar(r=r, rs=rs)            # 150 pasos de Adam
        finales[(r, rs)] = hist
        nombre = "rsLoRA α/√r" if rs else "LoRA   α/r "
        print(f"r={r:2} {nombre}: paso 50 {hist[50]:.3f},",
              f"final {hist[-1]:.3f}")

r= 4 LoRA   α/r : paso 50 0.596, final 0.529
r= 4 rsLoRA α/√r: paso 50 0.575, final 0.520
r=16 LoRA   α/r : paso 50 0.576, final 0.354
r=16 rsLoRA α/√r: paso 50 0.456, final 0.258


r=64 LoRA   α/r : paso 50 0.567, final 0.250
r=64 rsLoRA α/√r: paso 50 0.262, final 0.013


In [6]:
# Verificación extra (corrió junto con el video)
json.dump({f"{r}_{int(rs)}": h for (r, rs), h in finales.items()}, open(os.path.join(DATOS, "ep09_rango.json"), "w"))

### Pérdida, rango 64

Las curvas. Rango sesenta y cuatro con alfa sobre r: baja despacio. El mismo rango con alfa sobre raíz de r: se desploma antes del paso cincuenta. Y la punteada, rango dieciséis con la escala común, va casi pegada a la roja: cuadruplicar el rango no compró nada. Eso es lo que el paper llama aprendizaje frenado en rangos altos. Y es una trampa fácil: subís el rango esperando más capacidad, no ves mejora, y concluís que el rango no importa. Lo que no servía era la escala.

## Parte 2 · LoRA+: dos learning rates

A y B no cumplen el mismo papel: B arranca en cero y A al azar. ¿Por qué deberían aprender al mismo ritmo?

### B, λ veces más rápido

LoRA+ es una línea en el optimizador. Un grupo de parámetros para A, con el learning rate de siempre. Otro para B, con lambda veces más. Probamos lambda uno, que es LoRA común, cuatro y dieciséis. Rango dieciséis, la misma tarea. Con lambda uno, paso cincuenta cero coma cinco siete seis, final cero coma tres cinco cuatro. Con lambda dieciséis, paso cincuenta cero coma tres cinco ocho, final cero coma dos tres nueve. Y fijate en lambda cuatro: cero coma cuatro cinco seis y cero coma dos cinco ocho. Son exactamente los números de rsLoRA con rango dieciséis. No es casualidad.

In [7]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep09_loraplus.py
__file__ = 'snippets/ep09_loraplus.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import json
import os
import torch
import torch.nn as nn
from peft import LoraConfig, get_peft_model
d = 64
DATOS = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "datos")


class Capa(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Linear(d, d, bias=False)

    def forward(self, x):
        return self.proj(x)


def tarea(tipo="azar"):
    """Base al azar y una tarea sintética: un cambio al azar de rango completo, o reescalar las filas de W."""
    torch.manual_seed(0)
    base = Capa()
    W0 = base.proj.weight.detach()
    g = torch.Generator().manual_seed(123)
    cambio = torch.randn(d, d, generator=g) * 0.1
    X = torch.randn(512, d, generator=g)
    if tipo == "azar":
        Y = X @ (W0 + cambio).T
    else:                                       # cada fila de W, estirada o encogida
        escala = torch.exp(0.3 * torch.randn(d, 1, generator=g))   # siempre > 0
        Y = X @ (escala * W0).T
    return base, X, Y


def entrenar(r=16, rs=False, dora=False, lam=1.0, lr=1e-3, pasos=150, tipo="azar"):
    base, X, Y = tarea(tipo)
    cfg = LoraConfig(r=r, lora_alpha=8, target_modules=["proj"], use_rslora=rs, use_dora=dora)
    m = get_peft_model(base, cfg)
    A = [p for n, p in m.named_parameters() if p.requires_grad and "lora_B" not in n]
    B = [p for n, p in m.named_parameters() if "lora_B" in n]
    opt = torch.optim.Adam([{"params": A, "lr": lr}, {"params": B, "lr": lam * lr}])
    hist = []
    for _ in range(pasos):
        perdida = ((m(X) - Y) ** 2).mean()
        opt.zero_grad()
        perdida.backward()
        opt.step()
        hist.append(perdida.item())
    return hist

In [8]:
# LoRA+: B aprende λ veces más rápido que A
def grupos(modelo, lr, lam):
    A = [p for n, p in modelo.named_parameters() if "lora_A" in n]
    B = [p for n, p in modelo.named_parameters() if "lora_B" in n]
    return [{"params": A, "lr": lr}, {"params": B, "lr": lam * lr}]

curvas = {}
for lam in [1, 4, 16]:                       # λ = 1 es LoRA común
    curvas[lam] = entrenar(r=16, lam=lam)
    h = curvas[lam]
    print(f"λ = {lam:2}: paso 25 {h[25]:.3f}, paso 50 {h[50]:.3f},",
          f"final {h[-1]:.3f}")

λ =  1: paso 25 0.627, paso 50 0.576, final 0.354
λ =  4: paso 25 0.561, paso 50 0.456, final 0.258


λ = 16: paso 25 0.453, paso 50 0.358, final 0.239


In [9]:
# Verificación extra (corrió junto con el video)
json.dump(curvas, open(os.path.join(DATOS, "ep09_loraplus.json"), "w"))

### Pérdida según λ (r = 16)

LoRA común, punteada. Lambda cuatro, más rápido. Lambda dieciséis, todavía más. En este juguete, lambda dieciséis llega en cincuenta pasos a donde la común llega en ciento cincuenta. La explicación del paper, para modelos muy anchos: usar el mismo learning rate en A y en B hace que el aprendizaje sea ineficiente, y la corrección es darle a B uno mayor.

### Con Adam, escala y lr de B se compensan

$$
\text{Adam}: \; \lvert \Delta \theta \rvert \approx \eta
$$
<small>$\theta$: cada parámetro · $\eta$: learning rate</small>

$$
\gamma\, B_t \approx \gamma\, \eta_B\, t
$$
<small>lo que importa es el producto γ · η_B</small>

$$
\text{rsLoRA}_{r=16} \;\approx\; \text{LoRA+}_{\lambda = 4}
$$
<small>$\alpha/\sqrt{16} = 4 \cdot \alpha/16$</small>

¿Por qué coinciden? Adam divide cada gradiente por un promedio de su propio tamaño. Resultado: cada parámetro, theta, se mueve en cada paso más o menos eta, el learning rate, sin importar si el gradiente es grande o chico. En el dibujo, gradiente por cuatro, mismo paso. B arranca en cero y crece más o menos eta sub B por paso; el desvío es gamma por B. Lo que cuenta es el producto. Y A no se entera, porque Adam también le normaliza su gradiente. Con r dieciséis, rsLoRA multiplica la escala por cuatro: es lo mismo que LoRA+ con lambda cuatro. Por eso los números son idénticos.

> **Hayou, Ghosh y Yu, 2024 — LoRA+: Efficient Low Rank Adaptation of Large Models** · [arXiv:2402.12354](https://arxiv.org/abs/2402.12354)
>
> **hasta ~2× más rápido** · «LoRA+ improves performance (1-2 % improvements) and finetuning speed (up to ~ 2X SpeedUp), at the same computational cost as LoRA.»

El paper de LoRA+ reporta entre uno y dos puntos de mejora y hasta el doble de velocidad, al mismo costo. El lambda óptimo no es universal: con su inicialización, recomiendan alrededor de dos a la cuarta, dieciséis, para RoBERTa, y entre dos y cuatro para Llama. Y aclaran que depende del modelo y de la tarea.

## Parte 3 · DoRA: magnitud y dirección

Cada fila de W es un vector: tiene un largo y una dirección. DoRA las entrena por separado.

### Una fila de W: largo y dirección

Un esquema en dos dimensiones. Una fila de la matriz original, W sub cero. LoRA le suma un cambio: la punta se mueve. Llamemos V a la fila resultante: girada respecto de la original. DoRA se queda solo con la dirección de V y le pone el largo que decida un número entrenable, m. En el dibujo, m encoge la fila sin tocar su dirección. Con LoRA común, cambiar el largo de una fila sin girarla exige un cambio que se parezca a la fila entera.

<small>esquema en 2D, valores inventados</small>

### DoRA, en fórmulas

$$
m_i = \lVert W_{0,i} \rVert
$$
<small>un número por fila: $d$ parámetros más</small>

$$
V = W_0 + s\, B A
$$
<small>la parte de LoRA: decide la dirección</small>

$$
W'_i = m_i\, \frac{V_i}{\lVert V_i \rVert}
$$
<small>fila $i$: largo $m_i$, dirección de $V_i$</small>

Al empezar, m sub i es el largo de la fila i de W sub cero: su norma, la raíz de la suma de sus cuadrados. Es un vector de d números, uno por fila, y entrena. En el dibujo, un largo por fila. V es W sub cero más s por B por A, LoRA de siempre: s es la escala. Cada fila de la matriz final es m sub i por V sub i dividido por su norma: el largo lo pone m, la dirección la pone V. En el dibujo, si m creció uno coma cinco veces, cada fila mide eso, apunte V adonde apunte. Un detalle: el paper habla de columnas; peft, con la matriz guardada como salidas por entradas, lo hace por filas.

### DoRA a mano contra peft

Con use dora en verdadero, sobre una capa de ocho por ocho. peft guarda el vector de magnitud en la capa. Comprobamos que arranca igual al largo de cada fila de W sub cero. Después lo cambiamos, y B también, como si hubiéramos entrenado. V, la parte de LoRA. La matriz de DoRA, fila por fila. Y comparamos con la salida de peft. La magnitud tiene ocho números y arranca en el largo de cada fila. Las filas de nuestra matriz miden exactamente la magnitud. Con peft difiere en uno coma ocho por diez a la menos siete. Y entrenan cuarenta números: dieciséis de A, dieciséis de B y ocho de magnitud.

In [10]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep09_dora.py
__file__ = 'snippets/ep09_dora.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import torch
import torch.nn as nn
from peft import LoraConfig, get_peft_model
torch.manual_seed(0)


class Capa(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Linear(8, 8, bias=False)

    def forward(self, x):
        return self.proj(x)


m = get_peft_model(Capa(), LoraConfig(r=2, lora_alpha=4, target_modules=["proj"], use_dora=True))
q = m.base_model.model.proj
x = torch.randn(5, 8)

In [11]:
W0 = q.base_layer.weight                          # d × k, congelada
A, B = q.lora_A["default"].weight, q.lora_B["default"].weight
s = q.scaling["default"]
mag = q.lora_magnitude_vector["default"].weight   # d números, entrena
print("magnitud:", tuple(mag.shape))
print("inicio = normas de W0:",
      torch.allclose(mag, W0.norm(dim=1)))
with torch.no_grad():                             # como tras entrenar
    mag.mul_(1.5)
    B.normal_()

V = W0 + s * B @ A                                # la dirección, sin normalizar
W_dora = mag[:, None] * V / V.norm(dim=1, keepdim=True)
print("normas de W_dora = mag:",
      torch.allclose(W_dora.norm(dim=1), mag))
print(f"máx |peft − mía|: {(q(x) - x @ W_dora.T).abs().max():.1e}")
print("entrenables:", sum(p.numel() for p in m.parameters() if p.requires_grad))

magnitud: (8,)
inicio = normas de W0: True
normas de W_dora = mag: True
máx |peft − mía|: 1.8e-07
entrenables: 40


> **Liu et al., 2024 — DoRA: Weight-Decomposed Low-Rank Adaptation** · [arXiv:2402.09353](https://arxiv.org/abs/2402.09353)
>
> **magnitud + dirección** · «DoRA decomposes the pre-trained weight into two components, magnitude and direction, for fine-tuning».

La motivación del paper es una medición: en el fine-tuning completo, el cambio de largo y el de dirección muestran una relación más bien negativa; en LoRA, en cambio, aparecen atados, con una correlación positiva de cero coma ochenta y tres. DoRA los desata con un vector más. Y sugiere tratar la norma de V como constante en el backward para ahorrar memoria: es lo que hace peft.

## Parte 4 · Las cuatro, en la misma tarea

Dos tareas sintéticas: un cambio al azar, y estirar o encoger cada fila de W.

### LoRA, rsLoRA, LoRA+ y DoRA

Las cuatro con rango dieciséis. rsLoRA. LoRA+ con lambda dieciséis. DoRA. Esta vez, seiscientos pasos, para ver adónde llegan y no solo qué tan rápido, en dos tareas: el cambio al azar de antes, y otra que estira o encoge cada fila de W por un factor distinto. En la tarea al azar, todas terminan cerca de cero coma veintidós: es el mejor error posible con rango dieciséis, y rsLoRA y LoRA+ solo llegaban antes. En la de filas, LoRA, rsLoRA y LoRA+ se clavan en cero coma cero cero nueve tres. DoRA baja a cero coma cero cero cero nueve: diez veces menos.

In [12]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep09_todas.py
__file__ = 'snippets/ep09_todas.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import json
import os
import torch
import torch.nn as nn
from peft import LoraConfig, get_peft_model
d = 64
DATOS = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "datos")


class Capa(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Linear(d, d, bias=False)

    def forward(self, x):
        return self.proj(x)


def tarea(tipo="azar"):
    """Base al azar y una tarea sintética: un cambio al azar de rango completo, o reescalar las filas de W."""
    torch.manual_seed(0)
    base = Capa()
    W0 = base.proj.weight.detach()
    g = torch.Generator().manual_seed(123)
    cambio = torch.randn(d, d, generator=g) * 0.1
    X = torch.randn(512, d, generator=g)
    if tipo == "azar":
        Y = X @ (W0 + cambio).T
    else:                                       # cada fila de W, estirada o encogida
        escala = torch.exp(0.3 * torch.randn(d, 1, generator=g))   # siempre > 0
        Y = X @ (escala * W0).T
    return base, X, Y


def entrenar(r=16, rs=False, dora=False, lam=1.0, lr=1e-3, pasos=150, tipo="azar"):
    base, X, Y = tarea(tipo)
    cfg = LoraConfig(r=r, lora_alpha=8, target_modules=["proj"], use_rslora=rs, use_dora=dora)
    m = get_peft_model(base, cfg)
    A = [p for n, p in m.named_parameters() if p.requires_grad and "lora_B" not in n]
    B = [p for n, p in m.named_parameters() if "lora_B" in n]
    opt = torch.optim.Adam([{"params": A, "lr": lr}, {"params": B, "lr": lam * lr}])
    hist = []
    for _ in range(pasos):
        perdida = ((m(X) - Y) ** 2).mean()
        opt.zero_grad()
        perdida.backward()
        opt.step()
        hist.append(perdida.item())
    return hist

In [13]:
variantes = {
    "LoRA":   dict(),
    "rsLoRA": dict(rs=True),
    "LoRA+":  dict(lam=16),
    "DoRA":   dict(dora=True),
}
finales = {}
print("pérdida, 600 pasos  al azar   filas")
for nombre, opciones in variantes.items():
    finales[nombre] = [entrenar(r=16, pasos=600, tipo=t, **opciones)[-1]
                       for t in ["azar", "filas"]]
    print(f"{nombre:18} {finales[nombre][0]:7.4f} {finales[nombre][1]:7.4f}")

pérdida, 600 pasos  al azar   filas


LoRA                0.2279  0.0093


rsLoRA              0.2264  0.0093


LoRA+               0.2253  0.0093


DoRA                0.2189  0.0009


In [14]:
# Verificación extra (corrió junto con el video)
json.dump(finales, open(os.path.join(DATOS, "ep09_todas.json"), "w"))

### Estirar filas: pérdida final (r = 16)

La tarea de estirar filas, en barras. LoRA. rsLoRA. LoRA+: las tres se clavan en el mismo valor. DoRA, diez veces más abajo. ¿Por qué? Estirar cada fila por un factor distinto es un cambio de rango completo, y rango dieciséis no lo alcanza; para DoRA, en cambio, es mover un número por fila: la magnitud. En la tarea al azar, en cambio, las cuatro empatan.

- **LoRA**: 0,0093
- **rsLoRA**: 0,0093
- **LoRA+**: 0,0093
- **DoRA**: 0,0009

### Cuál usar

- rsLoRA: use_rslora=True si subís el rango
- LoRA+: create_loraplus_optimizer, λ entre 2 y 16
- DoRA: use_dora=True; un forward más caro
- Con Adam, escala y lr de B se compensan

En la práctica. Si vas a usar rangos altos, use rslora en verdadero: no cuesta nada. LoRA+ viene en peft como create loraplus optimizer, y el lambda hay que barrerlo. DoRA es use dora en verdadero; suma d parámetros por capa y un forward más caro, porque hay que normalizar la matriz. Y lo de hoy: con Adam, la escala y el learning rate de B se compensan; antes de comparar variantes, mirá si no estás comparando learning rates.

## Resumiendo

- rsLoRA: $\gamma = \alpha/\sqrt{r}$; con $\alpha/r$, el rango alto aprende lento
- LoRA+: $\eta_B = \lambda\, \eta_A$, $\lambda$ = 16 fue 3× más rápido acá
- Con Adam: escala $\times c$ ≈ lr de B $\times c$
- DoRA: $W' = m \cdot V/\lVert V \rVert$, un largo por fila
- DoRA: 10× menos error al estirar filas
- Próximo: qué aprende y qué olvida LoRA